In [ ]:
import pandas as pd
from datetime import datetime

def process_conversation_data(original_df):
    """
    Process conversation data into a structured DataFrame with conversation_id,
    date, time, question and answer columns.
    
    Args:
        original_df (pd.DataFrame): DataFrame containing the mapping column
    Returns:
        pd.DataFrame: Processed conversation data
    """
    # Initialize lists to store the processed data
    processed_data = []
    
    # Iterate through each row in the original DataFrame
    for _, row in original_df.iterrows():
        conversation_id = row["id"]
        title = row["title"]
        mapping = row["mapping"]
        conversation_data = list(mapping.values())
        
        # Iterate through messages with index to check next message
        for i, message in enumerate(conversation_data):
            # Check if the message has required fields and is from a user
            if (message.get('message') and
                message['message'].get('author') and
                message['message']['author'].get('role') == 'user' and
                message['message'].get('create_time') and
                message['message'].get('content')):
                
                # Extract timestamp and convert to datetime
                timestamp = message['message']['create_time']
                dt = datetime.fromtimestamp(timestamp)
                
                # Process question
                question_parts = message['message']['content'].get('parts', [''])
                question_parts = [str(part) if isinstance(part, dict) else part for part in question_parts]
                question = "\n".join(question_parts)
                
                # Get answer from next message if it exists and is from assistant
                answer = ""
                if i + 1 < len(conversation_data):
                    next_message = conversation_data[i + 1]
                    if (next_message.get('message') and
                        next_message['message'].get('author') and
                        next_message['message']['author'].get('role') == 'assistant' and
                        next_message['message'].get('content')):
                        answer_parts = next_message['message']['content'].get('parts', [''])
                        answer_parts = [str(part) if isinstance(part, dict) else part for part in answer_parts]
                        answer = "\n".join(answer_parts)
                
                processed_data.append({
                    'conversation_id': conversation_id,
                    'title': title,
                    'date': dt.date(),
                    'time': dt.time(),
                    'question': question,
                    'answer': answer
                })
    
    # Create DataFrame from processed data
    df_conversations = pd.DataFrame(processed_data)
    
    # Sort by date and time
    df_conversations = df_conversations.sort_values(['date', 'time'])
    
    return df_conversations

In [ ]:
from pathlib import Path
import pandas as pd
from dtale import show
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / "data"
df = pd.read_json(data_dir / "conversations/cm0i27jdj0000aqpa73ghpcxf.json")
df_processed = process_conversation_data(df)



In [ ]:
show(df_processed).open_browser()

In [ ]:
from io import StringIO
import polars as pl

csv_file = StringIO()
df_processed.to_csv(csv_file, index=False)
csv_file.seek(0)


grouped_df = pl.read_csv(csv_file).sort("date", "time").with_columns(
    pl.concat_str(
        [
            pl.col("date"),
            pl.lit(" at "),
            pl.col("time"),
            pl.lit(" : '''"),
            pl.col("content"),
            pl.lit("'''")
        ]
    ).alias("timed_contents")
).group_by(['conversation_id', 'title']).agg([
    pl.col("timed_contents").str.concat("\n"),
    pl.col("date").first()
]
)

In [ ]:
show(grouped_df.to_pandas()).open_browser()
